# Stap 5 — OSMnx Rijafstanden

Deze cellen vervangen de Haversine luchtlijn afstand door echte afstanden
over de wegenkaart van Heerlen via OSMnx.

Voeg deze cellen toe aan algorithm.ipynb ter vervanging van cel 5.

In [ ]:
# Installeer indien nodig
# %pip install osmnx networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import osmnx as ox
import networkx as nx
import os
import pickle

print(f'OSMnx versie: {ox.__version__}')

OSMnx versie: 2.1.0


## 5a. Wegenkaart Heerlen downloaden

We downloaden de wegenkaart twee keer:
- `drive` netwerk voor autorijden
- `bike` netwerk voor fietsen

De kaarten worden lokaal gecached zodat je dit maar eenmalig hoeft te doen.

In [3]:
CACHE_DIR  = '../data/osm_cache'
CACHE_AUTO = os.path.join(CACHE_DIR, 'heerlen_drive.pkl')
CACHE_FIETS = os.path.join(CACHE_DIR, 'heerlen_bike.pkl')

os.makedirs(CACHE_DIR, exist_ok=True)

def laad_of_download_kaart(cache_pad, netwerk_type, plaatsnaam='Heerlen, Netherlands'):
    """Laadt kaart uit cache of download via OSMnx."""
    if os.path.exists(cache_pad):
        print(f'  Kaart geladen uit cache: {cache_pad}')
        with open(cache_pad, 'rb') as f:
            return pickle.load(f)
    else:
        print(f'  Downloading {netwerk_type} kaart van {plaatsnaam}...')
        G = ox.graph_from_place(plaatsnaam, network_type=netwerk_type)
        # Voeg reistijden toe aan edges
        if netwerk_type == 'drive':
            G = ox.add_edge_speeds(G)
        G = ox.add_edge_travel_times(G)
        with open(cache_pad, 'wb') as f:
            pickle.dump(G, f)
        print(f'  Gecached naar: {cache_pad}')
        return G

print('Auto netwerk laden...')
G_auto = laad_of_download_kaart(CACHE_AUTO, 'drive')
print(f'  Nodes: {len(G_auto.nodes):,}  |  Edges: {len(G_auto.edges):,}')

print('Fiets netwerk laden...')
G_fiets = laad_of_download_kaart(CACHE_FIETS, 'bike')
print(f'  Nodes: {len(G_fiets.nodes):,}  |  Edges: {len(G_fiets.edges):,}')

print('\nKaarten klaar!')

Auto netwerk laden...
  Gecached naar: ../data/osm_cache\heerlen_drive.pkl
  Nodes: 3,617  |  Edges: 8,584
Fiets netwerk laden...


KeyError: "All edges must have 'length' and 'speed_kph' attributes."

## 5b. Coordinaten koppelen aan wegennetwerk

Elk GPS coordinaat wordt gekoppeld aan de dichtstbijzijnde knoop
in het wegennetwerk (nearest node).

In [ ]:
def vind_nearest_nodes(G, coords_lijst):
    """
    Koppelt een lijst van (lat, lon) coordinaten aan de
    dichtstbijzijnde knopen in het wegennetwerk.

    Returns:
        lijst van node IDs (zelfde volgorde als coords_lijst)
    """
    lats = [c[0] for c in coords_lijst]
    lons = [c[1] for c in coords_lijst]
    # OSMnx verwacht (lon, lat) volgorde
    nodes = ox.nearest_nodes(G, lons, lats)
    return list(nodes)

# Alle coordinaten samenvoegen
alle_coords = list(df_mw_ok['coords']) + list(df_cl_ok['coords'])
n_mw = len(df_mw_ok)
n_cl = len(df_cl_ok)

print('Nearest nodes zoeken voor auto netwerk...')
nodes_auto  = vind_nearest_nodes(G_auto,  alle_coords)
print(f'  {len(nodes_auto)} nodes gevonden')

print('Nearest nodes zoeken voor fiets netwerk...')
nodes_fiets = vind_nearest_nodes(G_fiets, alle_coords)
print(f'  {len(nodes_fiets)} nodes gevonden')

## 5c. Afstandsmatrix berekenen via OSMnx

We berekenen twee matrices:
- `matrix_auto`  — rijafstand in meters
- `matrix_fiets` — fietsafstand in meters

Als een route niet bereikbaar is (disconnected graph), vallen we terug op Haversine * 1.4 als schatting.

In [ ]:
def haversine(coord1, coord2):
    """Fallback: luchtlijn afstand in meters."""
    import math
    R = 6371000
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def maak_osm_afstandsmatrix(G, nodes, alle_coords, gewicht='length'):
    """
    Bouwt NxN afstandsmatrix via kortste paden in OSMnx.

    Parameters:
        G         : OSMnx graaf
        nodes     : lijst van node IDs per punt
        alle_coords: originele coordinaten (voor fallback)
        gewicht   : 'length' voor meters, 'travel_time' voor seconden

    Returns:
        NxN matrix met afstanden als integers
    """
    n = len(nodes)
    matrix = [[0] * n for _ in range(n)]
    niet_bereikbaar = 0

    for i in range(n):
        # Bereken kortste paden vanuit node i naar alle andere nodes
        try:
            lengtes = nx.single_source_dijkstra_path_length(
                G, nodes[i], weight=gewicht
            )
        except nx.NodeNotFound:
            lengtes = {}

        for j in range(n):
            if i == j:
                continue
            if nodes[j] in lengtes:
                matrix[i][j] = int(lengtes[nodes[j]])
            else:
                # Fallback: haversine * 1.4 (omrijfactor)
                matrix[i][j] = int(haversine(alle_coords[i], alle_coords[j]) * 1.4)
                niet_bereikbaar += 1

    if niet_bereikbaar > 0:
        print(f'  Let op: {niet_bereikbaar} paren niet bereikbaar, fallback gebruikt')

    return matrix


# Auto matrix (afstand in meters)
print('Auto afstandsmatrix berekenen...')
matrix_auto = maak_osm_afstandsmatrix(G_auto, nodes_auto, alle_coords, gewicht='length')
print(f'  Klaar! Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {matrix_auto[0][n_mw]:.0f} m')

# Fiets matrix (afstand in meters)
print('Fiets afstandsmatrix berekenen...')
matrix_fiets = maak_osm_afstandsmatrix(G_fiets, nodes_fiets, alle_coords, gewicht='length')
print(f'  Klaar! Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {matrix_fiets[0][n_mw]:.0f} m')

# Haversine matrix voor vergelijking
print('Luchtlijn matrix berekenen (vergelijking)...')
matrix_luchtlijn = [[0]*len(alle_coords) for _ in range(len(alle_coords))]
for i in range(len(alle_coords)):
    for j in range(len(alle_coords)):
        if i != j:
            matrix_luchtlijn[i][j] = int(haversine(alle_coords[i], alle_coords[j]))
print('  Klaar!')

## 5d. Vergelijking: luchtlijn vs auto vs fiets

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Verzamel alle paar-afstanden (alleen mw -> client)
luchtlijn_afstanden = []
auto_afstanden      = []
fiets_afstanden     = []
labels              = []

for mw_idx in range(n_mw):
    for cl_idx in range(n_cl):
        node_cl = n_mw + cl_idx
        luchtlijn_afstanden.append(matrix_luchtlijn[mw_idx][node_cl] / 1000)
        auto_afstanden.append(matrix_auto[mw_idx][node_cl] / 1000)
        fiets_afstanden.append(matrix_fiets[mw_idx][node_cl] / 1000)
        labels.append(f"{df_mw_ok.iloc[mw_idx]['name']} -> {df_cl_ok.iloc[cl_idx]['Naam']}")

# Omrijfactoren
factor_auto  = np.mean([a/l for a, l in zip(auto_afstanden,  luchtlijn_afstanden) if l > 0])
factor_fiets = np.mean([f/l for f, l in zip(fiets_afstanden, luchtlijn_afstanden) if l > 0])

print('=' * 50)
print('VERGELIJKING AFSTANDSMETHODEN')
print('=' * 50)
print(f'Gemiddelde afstand luchtlijn : {np.mean(luchtlijn_afstanden):.2f} km')
print(f'Gemiddelde afstand auto      : {np.mean(auto_afstanden):.2f} km')
print(f'Gemiddelde afstand fiets     : {np.mean(fiets_afstanden):.2f} km')
print()
print(f'Omrijfactor auto  : {factor_auto:.2f}x  (auto is gem. {(factor_auto-1)*100:.0f}% langer dan luchtlijn)')
print(f'Omrijfactor fiets : {factor_fiets:.2f}x  (fiets is gem. {(factor_fiets-1)*100:.0f}% langer dan luchtlijn)')

# Grafiek
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram van afstanden
axes[0].hist(luchtlijn_afstanden, bins=30, alpha=0.6, label='Luchtlijn', color='gray')
axes[0].hist(auto_afstanden,      bins=30, alpha=0.6, label='Auto',      color='#2563eb')
axes[0].hist(fiets_afstanden,     bins=30, alpha=0.6, label='Fiets',     color='#16a34a')
axes[0].set_xlabel('Afstand (km)')
axes[0].set_ylabel('Aantal paren')
axes[0].set_title('Verdeling afstanden per methode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: luchtlijn vs auto
axes[1].scatter(luchtlijn_afstanden, auto_afstanden,  alpha=0.3, s=15, color='#2563eb', label='Auto')
axes[1].scatter(luchtlijn_afstanden, fiets_afstanden, alpha=0.3, s=15, color='#16a34a', label='Fiets')
max_val = max(max(luchtlijn_afstanden), max(auto_afstanden), max(fiets_afstanden))
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='1:1 lijn')
axes[1].set_xlabel('Luchtlijn (km)')
axes[1].set_ylabel('Werkelijke afstand (km)')
axes[1].set_title('Luchtlijn vs werkelijke afstand')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Vergelijking afstandsmethoden — Heerlen', fontweight='bold')
plt.tight_layout()
plt.savefig('../output/afstand_vergelijking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafiek opgeslagen in output/afstand_vergelijking.png')

## 5e. Kies matrix voor OR-Tools

Kies welke matrix je wil gebruiken voor de route optimalisatie.
Verander `TRANSPORT_MODUS` naar `'auto'` of `'fiets'`.

In [ ]:
# --- KEUZE: 'auto' of 'fiets' ---
TRANSPORT_MODUS = 'auto'
# --------------------------------

if TRANSPORT_MODUS == 'auto':
    afstand_matrix = matrix_auto
    print('Modus: AUTO')
elif TRANSPORT_MODUS == 'fiets':
    afstand_matrix = matrix_fiets
    print('Modus: FIETS')
else:
    raise ValueError(f'Onbekende modus: {TRANSPORT_MODUS}. Kies auto of fiets.')

print(f'Afstandsmatrix klaar voor OR-Tools: {len(afstand_matrix)}x{len(afstand_matrix[0])}')
print('Ga nu verder met cel 6: OR-Tools VRP oplossen')